# MuleNet: Multi-Task Causal GNN & XGBoost Hybrid for Money Mule Detection

This notebook trains the 107-feature MuleNet pipeline on the IBM Transactions for AML dataset (`HI-Small` split with `LI-Small` cross-distribution evaluation), exports bit-identical ONNX model artifacts, and benchmarks real-time sub-millisecond inference.


In [ ]:
# Step 1: Install prerequisite packages on Kaggle
!pip install -q torch-geometric onnx onnxmltools onnxruntime


In [ ]:
# Step 2: Unpack the upgraded MuleNet package (107 features, GNN, O(1) state store)
import os
from pathlib import Path

os.makedirs("mulenet", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)
os.makedirs("data/check", exist_ok=True)
os.makedirs("reports", exist_ok=True)

modules = {
  "__init__.py": "\"\"\"mulenet: laundering transaction scoring and mule-account roll-up for the IBM AML dataset.\"\"\"\n",
  "data.py": "\"\"\"Loading the IBM AML files and the fixed day-based split.\n\nThe loaders mirror the `functions` cell in mulenet.ipynb, which has to stay\nself-contained because the Kaggle kernel cannot import this package.\n\"\"\"\n\nfrom pathlib import Path\n\nimport pandas as pd\n\n# The raw header names both account columns \"Account\", so we supply our own names.\nCOLUMNS = [\n    \"timestamp\", \"from_bank\", \"from_account\", \"to_bank\", \"to_account\",\n    \"amount_received\", \"receiving_currency\", \"amount_paid\", \"payment_currency\",\n    \"payment_format\", \"is_laundering\",\n]\n# Bank codes have leading zeros (\"010\"), so keep ids as strings.\nDTYPES = {\n    \"from_bank\": str, \"from_account\": str, \"to_bank\": str, \"to_account\": str,\n    \"receiving_currency\": str, \"payment_currency\": str, \"payment_format\": str,\n    \"amount_received\": \"float64\", \"amount_paid\": \"float64\", \"is_laundering\": \"int8\",\n}\nTIME_FORMAT = \"%Y/%m/%d %H:%M\"\n\n# Whole-day split. After LAST_DAY the simulator only emits the tail of laundering\n# attempts (~58% positive), so those rows are dropped rather than scored.\nSPLITS = {\n    \"train\": (\"2022-09-01\", \"2022-09-06\"),\n    \"val\": (\"2022-09-07\", \"2022-09-08\"),\n    \"test\": (\"2022-09-09\", \"2022-09-10\"),\n}\nLAST_DAY = SPLITS[\"test\"][1]\n\n\ndef load_transactions(path) -> pd.DataFrame:\n    \"\"\"Load a *_Trans.csv file. Adds `txn_id`, the 0-based row number in the file.\"\"\"\n    df = pd.read_csv(path, header=0, names=COLUMNS, dtype=DTYPES)\n    df[\"timestamp\"] = pd.to_datetime(df[\"timestamp\"], format=TIME_FORMAT)\n    df.insert(0, \"txn_id\", range(len(df)))\n    return df\n\n\ndef trim_and_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:\n    \"\"\"Drop rows after LAST_DAY, sort by time, and label each row's split.\n\n    Returns (rows sorted by timestamp then txn_id, split name per row).\n    \"\"\"\n    day = df[\"timestamp\"].dt.strftime(\"%Y-%m-%d\")\n    df = df[day <= LAST_DAY].sort_values([\"timestamp\", \"txn_id\"], kind=\"stable\")\n    day = df[\"timestamp\"].dt.strftime(\"%Y-%m-%d\")\n    split = pd.Series(pd.NA, index=df.index, dtype=\"string\")\n    for name, (start, end) in SPLITS.items():\n        split[(day >= start) & (day <= end)] = name\n    assert split.notna().all(), \"rows outside every split\"\n    return df.reset_index(drop=True), split.reset_index(drop=True)\n\n\ndef trans_path(data_dir, prefix: str) -> Path:\n    return Path(data_dir) / f\"{prefix}_Trans.csv\"\n\n\ndef accounts_path(data_dir, prefix: str) -> Path:\n    return Path(data_dir) / f\"{prefix}_Accounts.csv\"\n\n\ndef patterns_path(data_dir, prefix: str) -> Path:\n    return Path(data_dir) / f\"{prefix}_Patterns.txt\"\n\n\nPATTERN_TYPES = (\n    \"FAN-OUT\",\n    \"FAN-IN\",\n    \"GATHER-SCATTER\",\n    \"SCATTER-GATHER\",\n    \"CYCLE\",\n    \"BIPARTITE\",\n    \"STACK\",\n    \"RANDOM\",\n)\n\n\ndef classify_entity_name(name: str) -> int:\n    \"\"\"Encode entity names into categorical codes:\n    0: Unknown\n    1: Individual / Person\n    2: Corporation / Company / LLC / Ltd / Inc / Corp\n    3: Sole Proprietorship\n    4: Partnership\n    5: Other / Organization\n    \"\"\"\n    if not isinstance(name, str) or not name.strip() or name.strip().lower() in (\"unknown\", \"none\", \"nan\", \"null\"):\n        return 0\n    n = name.lower()\n    if any(w in n for w in (\"corp\", \"inc\", \"llc\", \"ltd\", \"company\", \"co.\", \"gmbh\", \"s.a.\")):\n        return 2\n    if \"sole\" in n or \"proprietor\" in n:\n        return 3\n    if \"partner\" in n:\n        return 4\n    if \"org\" in n or \"foundation\" in n or \"trust\" in n or \"gov\" in n:\n        return 5\n    # Standard personal entity names map to Individual (1)\n    return 1\n\n\ndef load_accounts_metadata(path) -> dict[str, dict]:\n    \"\"\"Load an *_Accounts.csv file if it exists.\n\n    Maps account key (`f\"{bank}_{account}\"`) to:\n      {\"entity_id\": str, \"entity_type_code\": float}\n\n    Returns an empty dict if the file does not exist.\n    \"\"\"\n    p = Path(path)\n    if not p.is_file():\n        return {}\n\n    try:\n        sample = pd.read_csv(p, nrows=5)\n        col_map = {c.lower().replace(\" \", \"_\"): c for c in sample.columns}\n        bank_col = col_map.get(\"bank_id\", col_map.get(\"from_bank\", col_map.get(\"bank\", \"Bank ID\")))\n        acct_col = col_map.get(\"account_number\", col_map.get(\"from_account\", col_map.get(\"account\", \"Account Number\")))\n        entity_id_col = col_map.get(\"entity_id\", col_map.get(\"owner_id\", col_map.get(\"entity\", \"Entity ID\")))\n        entity_name_col = col_map.get(\"entity_name\", col_map.get(\"entity_type\", col_map.get(\"name\", \"Entity Name\")))\n\n        df = pd.read_csv(p, dtype=str)\n        out = {}\n        for r in df.to_dict(orient=\"records\"):\n            bank = str(r.get(bank_col, \"\")).strip()\n            acct = str(r.get(acct_col, \"\")).strip()\n            e_id = str(r.get(entity_id_col, \"\")).strip()\n            e_name = str(r.get(entity_name_col, \"\")).strip()\n            if bank and acct:\n                key = f\"{bank}_{acct}\"\n                code = float(classify_entity_name(e_name))\n                out[key] = {\"entity_id\": e_id, \"entity_type_code\": code}\n        return out\n    except Exception:\n        return {}\n\n\ndef parse_patterns_file(path) -> dict[tuple, str]:\n    \"\"\"Parse a *_Patterns.txt file into a mapping of transaction keys to pattern categories.\n\n    Keys: (timestamp, from_bank, from_account, to_bank, to_account, amount_paid_rounded)\n    Values: Pattern category ('FAN-OUT', 'CYCLE', etc.)\n    Returns empty dict if the file does not exist.\n    \"\"\"\n    p = Path(path)\n    if not p.is_file():\n        return {}\n\n    import re\n\n    pattern_map = {}\n    cur_pattern = None\n    with open(p, \"r\", encoding=\"utf-8\") as f:\n        for line in f:\n            line = line.strip()\n            if not line:\n                continue\n            m = re.match(r\"^BEGIN LAUNDERING ATTEMPT\\s*-\\s*([A-Za-z0-9\\-]+)\", line)\n            if m:\n                cur_pattern = m.group(1).upper()\n                continue\n            if line.startswith(\"END LAUNDERING ATTEMPT\"):\n                cur_pattern = None\n                continue\n            if cur_pattern:\n                parts = line.split(\",\")\n                if len(parts) >= 8:\n                    try:\n                        ts = pd.to_datetime(parts[0], format=TIME_FORMAT)\n                        from_b = str(parts[1]).strip()\n                        from_a = str(parts[2]).strip()\n                        to_b = str(parts[3]).strip()\n                        to_a = str(parts[4]).strip()\n                        amt = round(float(parts[7]), 2)\n                        key = (ts, from_b, from_a, to_b, to_a, amt)\n                        pattern_map[key] = cur_pattern\n                    except Exception:\n                        continue\n    return pattern_map\n",
  "features.py": "\"\"\"Transaction + account-graph features.\n\nEvery feature of a transaction at minute t uses only that transaction's own fields\nand transactions in the window [t - W, t), i.e. strictly earlier minutes. Labels are\nnever used, so features are identical at training and scoring time.\n\"\"\"\n\nimport numpy as np\nimport pandas as pd\n\nfrom .streaming import StreamingAccountStateStore\n\nWINDOWS = {\"6h\": 6 * 60, \"24h\": 24 * 60, \"7d\": 7 * 24 * 60}  # minutes\nMAX_WINDOW = max(WINDOWS.values())\n\nPAYMENT_FORMATS = [\"ACH\", \"Bitcoin\", \"Cash\", \"Cheque\", \"Credit Card\", \"Reinvestment\", \"Wire\"]\nCURRENCIES = [\n    \"Australian Dollar\", \"Bitcoin\", \"Brazil Real\", \"Canadian Dollar\", \"Euro\", \"Mexican Peso\",\n    \"Ruble\", \"Rupee\", \"Saudi Riyal\", \"Shekel\", \"Swiss Franc\", \"UK Pound\", \"US Dollar\", \"Yen\", \"Yuan\",\n]\n\nTXN_FEATURES = [\n    \"log_usd_paid\", \"log_usd_received\", \"usd_ratio\", \"currency_mismatch\",\n    \"payment_format\", \"payment_currency\", \"receiving_currency\",\n    \"hour\", \"weekday\", \"self_transfer\", \"same_bank\", \"whole_amount\",\n]\n# (prefix, which account, which direction of that account's flow)\nFLOWS = [\n    (\"src_out\", \"src\", \"out\"),  # sender's earlier payments\n    (\"src_in\", \"src\", \"in\"),    # money that reached the sender earlier (layering)\n    (\"dst_in\", \"dst\", \"in\"),    # receiver's earlier receipts (fan-in)\n    (\"dst_out\", \"dst\", \"out\"),  # receiver's earlier payments\n]\nFLOW_FEATURES = [\n    f\"{prefix}_{stat}_{w}\"\n    for prefix, _, _ in FLOWS\n    for w in WINDOWS\n    for stat in (\"cnt\", \"usd\", \"uniq\")\n] + [f\"{prefix}_mins_since_prev\" for prefix, _, _ in FLOWS]\nPAIR_FEATURES = [f\"pair_cnt_{w}\" for w in WINDOWS]\n\nINVARIANT_FEATURES = [\n    # Flow Throughput Ratios across 6h, 24h, 7d\n    \"src_throughput_ratio_6h\", \"src_throughput_ratio_24h\", \"src_throughput_ratio_7d\",\n    \"dst_throughput_ratio_6h\", \"dst_throughput_ratio_24h\", \"dst_throughput_ratio_7d\",\n    # Net Balance Retention Ratios across 24h, 7d\n    \"src_retention_ratio_24h\", \"src_retention_ratio_7d\",\n    \"dst_retention_ratio_24h\", \"dst_retention_ratio_7d\",\n    # Pass-Through Turnaround\n    \"src_passthrough_delta\", \"src_rapid_turnaround_flag\",\n    # Sleep & Wake Dormancy Spurt\n    \"src_dormancy_spurt\", \"dst_dormancy_spurt\",\n]\n\nCOLD_START_FEATURES = [\n    \"src_is_gnn_cold_start\",\n    \"dst_is_gnn_cold_start\",\n    \"both_cold_start\",\n]\n\nENTITY_FEATURES = [\n    \"src_entity_type\",\n    \"dst_entity_type\",\n    \"same_entity_transfer\",\n]\n\nGNN_DIM = 16\nGNN_FEATURES = [f\"src_gnn_{i}\" for i in range(GNN_DIM)] + [f\"dst_gnn_{i}\" for i in range(GNN_DIM)]\nFEATURES = TXN_FEATURES + FLOW_FEATURES + PAIR_FEATURES + INVARIANT_FEATURES + COLD_START_FEATURES + ENTITY_FEATURES + GNN_FEATURES\n\n_KEY_SPAN = np.int64(2**32)  # > any minute offset we will see; keeps key*span+t in int64\n\n\ndef fit_usd_rates(df: pd.DataFrame) -> dict[str, float]:\n    \"\"\"USD value of one unit of each currency, implied by cross-currency rows with a USD side.\"\"\"\n    rates = {\"US Dollar\": 1.0}\n    to_usd = df[(df[\"receiving_currency\"] == \"US Dollar\") & (df[\"payment_currency\"] != \"US Dollar\")]\n    from_usd = df[(df[\"payment_currency\"] == \"US Dollar\") & (df[\"receiving_currency\"] != \"US Dollar\")]\n    implied = pd.concat([\n        pd.DataFrame({\"cur\": to_usd[\"payment_currency\"],\n                      \"rate\": to_usd[\"amount_received\"] / to_usd[\"amount_paid\"]}),\n        pd.DataFrame({\"cur\": from_usd[\"receiving_currency\"],\n                      \"rate\": from_usd[\"amount_paid\"] / from_usd[\"amount_received\"]}),\n    ])\n    implied = implied[np.isfinite(implied[\"rate\"]) & (implied[\"rate\"] > 0)]\n    rates.update(implied.groupby(\"cur\")[\"rate\"].median().astype(float).to_dict())\n    return rates\n\n\ndef account_keys(df: pd.DataFrame) -> tuple[pd.Series, pd.Series]:\n    \"\"\"Account ids are only unique together with their bank.\"\"\"\n    return df[\"from_bank\"] + \"_\" + df[\"from_account\"], df[\"to_bank\"] + \"_\" + df[\"to_account\"]\n\n\ndef _window_stats(ev_key, ev_t, ev_vals, q_key, q_t):\n    \"\"\"For each query (key, t): stats over events with the same key and time in [t - W, t).\n\n    ev_vals maps a name to a per-event value to sum. Returns {f\"{name}_{w}\": sums,\n    f\"cnt_{w}\": counts, \"mins_since_prev\": minutes to the latest earlier event,\n    capped at MAX_WINDOW}.\n    \"\"\"\n    order = np.lexsort((ev_t, ev_key))\n    combined = ev_key[order] * _KEY_SPAN + ev_t[order]\n    sorted_t = ev_t[order]\n    q = q_key * _KEY_SPAN + q_t\n    hi = np.searchsorted(combined, q, side=\"left\")\n    cums = {name: np.concatenate([[0.0], np.cumsum(v[order], dtype=np.float64)])\n            for name, v in ev_vals.items()}\n\n    out = {}\n    for w, minutes in WINDOWS.items():\n        lo = np.searchsorted(combined, q - minutes, side=\"left\")\n        out[f\"cnt_{w}\"] = hi - lo\n        for name, cum in cums.items():\n            out[f\"{name}_{w}\"] = cum[hi] - cum[lo]\n\n    prev = np.maximum(hi - 1, 0)\n    has_prev = (hi > 0) & (combined[prev] // _KEY_SPAN == q_key)\n    since = np.where(has_prev, q_t - sorted_t[prev], MAX_WINDOW)\n    out[\"mins_since_prev\"] = np.minimum(since, MAX_WINDOW)\n    return out\n\n\ndef _new_counterparty_flags(acct, other, t) -> np.ndarray:\n    \"\"\"1 where this (acct, other) pair had no occurrence in the MAX_WINDOW before t.\n\n    Summing these flags over a window approximates the number of distinct\n    counterparties; it depends only on the past, so it is reproducible.\n    \"\"\"\n    pair = acct * _KEY_SPAN + other\n    order = np.lexsort((t, pair))\n    p, ts = pair[order], t[order]\n    prev_same = np.r_[False, p[1:] == p[:-1]]\n    gap = np.r_[MAX_WINDOW + 1, ts[1:] - ts[:-1]]\n    new_sorted = ~prev_same | (gap > MAX_WINDOW)\n    flags = np.empty(len(t), dtype=np.float64)\n    flags[order] = new_sorted\n    return flags\n\n\ndef build_features(\n    df: pd.DataFrame,\n    usd_rates: dict[str, float],\n    gnn_embeddings: dict[str, np.ndarray] = None,\n    accounts_metadata: dict[str, dict] = None,\n) -> pd.DataFrame:\n    \"\"\"One float32 feature row per transaction, aligned with df's index.\"\"\"\n    src, dst = account_keys(df)\n    codes, _ = pd.factorize(pd.concat([src, dst], ignore_index=True))\n    n = len(df)\n    src_c, dst_c = codes[:n].astype(np.int64), codes[n:].astype(np.int64)\n    t0 = pd.Timestamp(\"2022-01-01\")\n    t = ((df[\"timestamp\"] - t0) // pd.Timedelta(minutes=1)).to_numpy(np.int64)\n\n    paid_rate = df[\"payment_currency\"].map(usd_rates).astype(\"float64\").to_numpy()\n    recv_rate = df[\"receiving_currency\"].map(usd_rates).astype(\"float64\").to_numpy()\n    usd_paid = df[\"amount_paid\"].to_numpy() * paid_rate\n    usd_recv = df[\"amount_received\"].to_numpy() * recv_rate\n\n    f = {\n        \"log_usd_paid\": np.log1p(usd_paid),\n        \"log_usd_received\": np.log1p(usd_recv),\n        \"usd_ratio\": usd_recv / np.where(usd_paid > 0, usd_paid, np.nan),\n        \"currency_mismatch\": (df[\"payment_currency\"] != df[\"receiving_currency\"]).to_numpy(),\n        \"payment_format\": _codes(df[\"payment_format\"], PAYMENT_FORMATS),\n        \"payment_currency\": _codes(df[\"payment_currency\"], CURRENCIES),\n        \"receiving_currency\": _codes(df[\"receiving_currency\"], CURRENCIES),\n        \"hour\": df[\"timestamp\"].dt.hour.to_numpy(),\n        \"weekday\": df[\"timestamp\"].dt.weekday.to_numpy(),\n        \"self_transfer\": src_c == dst_c,\n        \"same_bank\": (df[\"from_bank\"] == df[\"to_bank\"]).to_numpy(),\n        \"whole_amount\": (df[\"amount_paid\"] == df[\"amount_paid\"].round()).to_numpy(),\n    }\n\n    usd = np.nan_to_num(usd_paid)\n    events = {\n        \"out\": (src_c, {\"usd\": usd, \"uniq\": _new_counterparty_flags(src_c, dst_c, t)}),\n        \"in\": (dst_c, {\"usd\": usd, \"uniq\": _new_counterparty_flags(dst_c, src_c, t)}),\n    }\n    query = {\"src\": src_c, \"dst\": dst_c}\n    for prefix, who, direction in FLOWS:\n        ev_key, ev_vals = events[direction]\n        stats = _window_stats(ev_key, t, ev_vals, query[who], t)\n        for name, values in stats.items():\n            f[f\"{prefix}_{name}\"] = values\n\n    pair = pd.factorize(src_c * _KEY_SPAN + dst_c)[0].astype(np.int64)\n    pair_stats = _window_stats(pair, t, {}, pair, t)\n    for w in WINDOWS:\n        f[f\"pair_cnt_{w}\"] = pair_stats[f\"cnt_{w}\"]\n\n    # --- Pattern-Agnostic Flow Invariants ---\n    for w in WINDOWS:\n        f[f\"src_throughput_ratio_{w}\"] = (f[f\"src_out_usd_{w}\"] + 1.0) / (f[f\"src_in_usd_{w}\"] + 1.0)\n        f[f\"dst_throughput_ratio_{w}\"] = (f[f\"dst_out_usd_{w}\"] + 1.0) / (f[f\"dst_in_usd_{w}\"] + 1.0)\n\n    for w in (\"24h\", \"7d\"):\n        f[f\"src_retention_ratio_{w}\"] = np.maximum(0.0, f[f\"src_in_usd_{w}\"] - f[f\"src_out_usd_{w}\"]) / (f[f\"src_in_usd_{w}\"] + 1.0)\n        f[f\"dst_retention_ratio_{w}\"] = np.maximum(0.0, f[f\"dst_in_usd_{w}\"] - f[f\"dst_out_usd_{w}\"]) / (f[f\"dst_in_usd_{w}\"] + 1.0)\n\n    f[\"src_passthrough_delta\"] = np.abs(f[\"src_out_mins_since_prev\"] - f[\"src_in_mins_since_prev\"])\n    f[\"src_rapid_turnaround_flag\"] = (f[\"src_in_mins_since_prev\"] < 60.0).astype(np.float32)\n\n    src_prior_daily_avg = np.maximum(0.0, f[\"src_out_usd_7d\"] - f[\"src_out_usd_24h\"]) / 6.0\n    f[\"src_dormancy_spurt\"] = (f[\"src_out_usd_24h\"] + 1.0) / (src_prior_daily_avg + 1.0)\n\n    dst_prior_daily_avg = np.maximum(0.0, f[\"dst_in_usd_7d\"] - f[\"dst_in_usd_24h\"]) / 6.0\n    f[\"dst_dormancy_spurt\"] = (f[\"dst_in_usd_24h\"] + 1.0) / (dst_prior_daily_avg + 1.0)\n\n    # --- Add GNN Embeddings & Cold-Start Indicators ---\n    if gnn_embeddings is not None:\n        src_cold = np.array([0.0 if k in gnn_embeddings else 1.0 for k in src], dtype=np.float32)\n        dst_cold = np.array([0.0 if k in gnn_embeddings else 1.0 for k in dst], dtype=np.float32)\n\n        def get_embs(keys):\n            emb_list = [gnn_embeddings.get(k, np.zeros(GNN_DIM, dtype=np.float32)) for k in keys]\n            return np.vstack(emb_list).astype(np.float32)\n\n        src_emb = get_embs(src)\n        dst_emb = get_embs(dst)\n    else:\n        src_cold = np.ones(n, dtype=np.float32)\n        dst_cold = np.ones(n, dtype=np.float32)\n        src_emb = np.zeros((n, GNN_DIM), dtype=np.float32)\n        dst_emb = np.zeros((n, GNN_DIM), dtype=np.float32)\n\n    f[\"src_is_gnn_cold_start\"] = src_cold\n    f[\"dst_is_gnn_cold_start\"] = dst_cold\n    f[\"both_cold_start\"] = src_cold * dst_cold\n\n    # --- Entity Ownership Features ---\n    if accounts_metadata:\n        src_types = np.array([float(accounts_metadata.get(k, {}).get(\"entity_type_code\", 0.0)) for k in src], dtype=np.float32)\n        dst_types = np.array([float(accounts_metadata.get(k, {}).get(\"entity_type_code\", 0.0)) for k in dst], dtype=np.float32)\n        src_eids = [accounts_metadata.get(k, {}).get(\"entity_id\", \"\") for k in src]\n        dst_eids = [accounts_metadata.get(k, {}).get(\"entity_id\", \"\") for k in dst]\n        same_e = np.array([\n            1.0 if (s and d and s == d) else 0.0\n            for s, d in zip(src_eids, dst_eids)\n        ], dtype=np.float32)\n    else:\n        src_types = np.zeros(n, dtype=np.float32)\n        dst_types = np.zeros(n, dtype=np.float32)\n        same_e = np.zeros(n, dtype=np.float32)\n\n    f[\"src_entity_type\"] = src_types\n    f[\"dst_entity_type\"] = dst_types\n    f[\"same_entity_transfer\"] = same_e\n\n    for i in range(GNN_DIM):\n        f[f\"src_gnn_{i}\"] = src_emb[:, i]\n        f[f\"dst_gnn_{i}\"] = dst_emb[:, i]\n\n    out = pd.DataFrame({name: np.asarray(f[name], dtype=np.float32) for name in FEATURES}, index=df.index)\n    return out\n\n\ndef _codes(values: pd.Series, vocab: list[str]) -> np.ndarray:\n    \"\"\"Fixed vocabulary codes; unseen values become -1.\"\"\"\n    return values.map({v: i for i, v in enumerate(vocab)}).fillna(-1).to_numpy()\n",
  "gnn.py": "import numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\nfrom torch_geometric.nn import SAGEConv\n\nclass GNNModel(torch.nn.Module):\n    def __init__(self, num_nodes, embedding_dim=16):\n        super().__init__()\n        self.node_emb = torch.nn.Embedding(num_nodes, embedding_dim)\n        self.conv1 = SAGEConv(embedding_dim, embedding_dim)\n        self.conv2 = SAGEConv(embedding_dim, embedding_dim)\n        self.classifier = torch.nn.Linear(embedding_dim * 2, 1)\n\n    def forward(self, edge_index, src, dst):\n        x = self.node_emb.weight\n        x = self.conv1(x, edge_index)\n        x = F.relu(x)\n        x = self.conv2(x, edge_index)\n        \n        edge_repr = torch.cat([x[src], x[dst]], dim=-1)\n        return self.classifier(edge_repr).squeeze(), x\n\ndef train_gnn(df_train: pd.DataFrame, embedding_dim=16, epochs=20, lr=0.01) -> dict[str, np.ndarray]:\n    \"\"\"Train a GNN on the train split and return account embeddings.\"\"\"\n    print(\"Training GNN...\")\n    \n    src_accounts = df_train[\"from_bank\"] + \"_\" + df_train[\"from_account\"]\n    dst_accounts = df_train[\"to_bank\"] + \"_\" + df_train[\"to_account\"]\n    all_accounts = pd.concat([src_accounts, dst_accounts]).unique()\n    \n    account_to_id = {acc: i for i, acc in enumerate(all_accounts)}\n    num_nodes = len(all_accounts)\n    \n    src_ids = src_accounts.map(account_to_id).values\n    dst_ids = dst_accounts.map(account_to_id).values\n    \n    edge_index = torch.tensor(np.vstack([src_ids, dst_ids]), dtype=torch.long)\n    y = torch.tensor(df_train[\"is_laundering\"].values, dtype=torch.float32)\n    \n    model = GNNModel(num_nodes, embedding_dim)\n    optimizer = torch.optim.Adam(model.parameters(), lr=lr)\n    \n    # Handle class imbalance (like XGBoost scale_pos_weight)\n    pos_weight = (len(y) - y.sum()) / max(y.sum(), 1.0)\n    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]))\n    \n    model.train()\n    for epoch in range(epochs):\n        optimizer.zero_grad()\n        out, node_embs = model(edge_index, edge_index[0], edge_index[1])\n        loss = criterion(out, y)\n        loss.backward()\n        optimizer.step()\n        if epoch % 5 == 0 or epoch == epochs - 1:\n            print(f\"GNN Epoch {epoch:02d}, Loss: {loss.item():.4f}\")\n            \n    model.eval()\n    with torch.no_grad():\n        _, final_embs = model(edge_index, edge_index[0], edge_index[1])\n        final_embs_np = final_embs.numpy()\n        \n    embeddings_dict = {acc: final_embs_np[i] for acc, i in account_to_id.items()}\n    print(f\"GNN Training complete. Extracted {len(embeddings_dict)} account embeddings.\")\n    return embeddings_dict\n",
  "model.py": "\"\"\"Training, metrics, account roll-up and model artifacts.\n\nEverything the scorer imports from here needs only numpy, pandas and one of\nxgboost / onnxruntime, so it can run on LinuxONE without scikit-learn.\n\"\"\"\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom .features import FEATURES\n\nPARAMS = {\n    \"objective\": \"binary:logistic\",\n    \"eval_metric\": \"aucpr\",\n    \"tree_method\": \"hist\",\n    \"max_depth\": 8,\n    \"eta\": 0.05,\n    \"subsample\": 0.8,\n    \"colsample_bytree\": 0.8,\n    \"min_child_weight\": 5,\n    \"scale_pos_weight\": 5,\n    \"seed\": 0,\n}\nNUM_ROUNDS = 2000\nEARLY_STOPPING = 100\n\n\n# ---------- metrics (numpy only) ----------\n\ndef pr_curve(y, p) -> tuple[np.ndarray, np.ndarray, np.ndarray]:\n    \"\"\"Precision, recall and threshold at every distinct score, highest score first.\n\n    Predicting positive means score >= threshold.\n    \"\"\"\n    y = np.asarray(y, dtype=np.float64)\n    p = np.asarray(p, dtype=np.float64)\n    order = np.argsort(-p, kind=\"stable\")\n    p, y = p[order], y[order]\n    last = np.r_[np.flatnonzero(p[1:] != p[:-1]), len(p) - 1]  # last index of each distinct score\n    tp = np.cumsum(y)[last]\n    fp = (last + 1) - tp\n    return tp / (tp + fp), tp / max(y.sum(), 1), p[last]\n\n\ndef average_precision(y, p) -> float:\n    \"\"\"Area under the PR curve as a step sum (same definition as sklearn).\"\"\"\n    precision, recall, _ = pr_curve(y, p)\n    return float(np.sum(np.diff(np.r_[0.0, recall]) * precision))\n\n\ndef best_f1_threshold(y, p) -> float:\n    precision, recall, thresholds = pr_curve(y, p)\n    f1 = np.where(precision + recall > 0, 2 * precision * recall / (precision + recall), 0.0)\n    return float(thresholds[np.argmax(f1)])\n\n\ndef classification_metrics(y, p, threshold: float) -> dict:\n    y = np.asarray(y).astype(bool)\n    pred = np.asarray(p) >= threshold\n    tp = int((pred & y).sum())\n    fp = int((pred & ~y).sum())\n    fn = int((~pred & y).sum())\n    precision = tp / (tp + fp) if tp + fp else 0.0\n    recall = tp / (tp + fn) if tp + fn else 0.0\n    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\n    return {\n        \"rows\": int(len(y)), \"positives\": int(y.sum()), \"alerts\": int(pred.sum()),\n        \"tp\": tp, \"fp\": fp, \"fn\": fn,\n        \"precision\": precision, \"recall\": recall, \"f1\": f1,\n        \"pr_auc\": average_precision(y, p),\n    }\n\n\ndef account_scores(src: pd.Series, dst: pd.Series, y, p) -> pd.DataFrame:\n    \"\"\"Roll transaction scores up to accounts.\n\n    score = max score over transactions the account sends or receives;\n    is_mule = the account touches at least one laundering transaction.\n    \"\"\"\n    both = pd.DataFrame({\n        \"account\": pd.concat([src, dst], ignore_index=True),\n        \"score\": np.r_[p, p],\n        \"is_laundering\": np.r_[y, y].astype(bool),\n    })\n    acc = both.groupby(\"account\").agg(\n        score=(\"score\", \"max\"), is_mule=(\"is_laundering\", \"any\"), txns=(\"score\", \"size\"),\n    )\n    return acc.sort_values(\"score\", ascending=False, kind=\"stable\")\n\n\ndef account_metrics(acc: pd.DataFrame, k: int = 100) -> dict:\n    top = acc.head(k)\n    return {\n        \"accounts\": int(len(acc)),\n        \"mules\": int(acc[\"is_mule\"].sum()),\n        \"pr_auc\": average_precision(acc[\"is_mule\"], acc[\"score\"]),\n        f\"precision_at_{k}\": float(top[\"is_mule\"].mean()),\n    }\n\n\n# ---------- training ----------\n\ndef train(X_train, y_train, X_val, y_val, log=print):\n    import xgboost as xgb\n\n    # Feature names come from the DataFrame columns (not the FEATURES constant), so this also\n    # works for an ablation that trains on a subset of FEATURES.\n    dtrain = xgb.DMatrix(X_train, label=y_train)\n    dval = xgb.DMatrix(X_val, label=y_val)\n    class Progress(xgb.callback.TrainingCallback):\n        def after_iteration(self, model, epoch, evals_log):\n            if epoch % 100 == 0:\n                log(f\"round {epoch}: val aucpr {evals_log['val']['aucpr'][-1]:.4f}\")\n            return False\n\n    booster = xgb.train(\n        PARAMS, dtrain, NUM_ROUNDS, evals=[(dval, \"val\")],\n        early_stopping_rounds=EARLY_STOPPING, verbose_eval=False, callbacks=[Progress()],\n    )\n    log(f\"best iteration {booster.best_iteration}, val aucpr {booster.best_score:.4f}\")\n    # Keep only the trees up to the best iteration so every backend scores identically.\n    return booster[: booster.best_iteration + 1]\n\n\n# ---------- artifacts ----------\n\ndef save_artifacts(out_dir, booster, meta: dict, gnn_embeddings: dict[str, np.ndarray] = None) -> None:\n    import onnxmltools\n    from onnxmltools.convert.common.data_types import FloatTensorType\n\n    out = Path(out_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    booster.save_model(out / \"model.json\")\n    # The ONNX converter wants f0..fN feature names.\n    onnx_booster = booster.copy()\n    onnx_booster.feature_names = None\n    onx = onnxmltools.convert_xgboost(\n        onnx_booster, initial_types=[(\"input\", FloatTensorType([None, len(FEATURES)]))], target_opset=15,\n    )\n    (out / \"model.onnx\").write_bytes(onx.SerializeToString())\n    (out / \"meta.json\").write_text(json.dumps(meta, indent=2) + \"\\n\")\n    if gnn_embeddings is not None:\n        keys = np.array(list(gnn_embeddings.keys()))\n        vals = np.array(list(gnn_embeddings.values()), dtype=np.float32)\n        np.savez_compressed(out / \"gnn_embeddings.npz\", keys=keys, values=vals)\n\n\ndef load_meta(artifacts_dir) -> dict:\n    return json.loads((Path(artifacts_dir) / \"meta.json\").read_text())\n\n\ndef load_gnn_embeddings(artifacts_dir) -> dict[str, np.ndarray]:\n    path = Path(artifacts_dir) / \"gnn_embeddings.npz\"\n    if not path.is_file():\n        return {}\n    data = np.load(path)\n    return dict(zip(data[\"keys\"], data[\"values\"]))\n\n\ndef load_predictor(artifacts_dir, backend: str = \"xgboost\"):\n    \"\"\"Return predict(X: float32 array [n, len(FEATURES)]) -> probability of laundering.\"\"\"\n    path = Path(artifacts_dir)\n    if backend == \"xgboost\":\n        import xgboost as xgb\n\n        booster = xgb.Booster()\n        booster.load_model(path / \"model.json\")\n        return lambda X: booster.predict(xgb.DMatrix(X, feature_names=FEATURES))\n    if backend == \"onnx\":\n        import os\n        import onnxruntime as ort\n\n        so = ort.SessionOptions()\n        so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL\n        # Configurable intra-op threads for s390x Telum co-processors\n        so.intra_op_num_threads = int(os.environ.get(\"ORT_INTRA_OP_NUM_THREADS\", min(4, os.cpu_count() or 1)))\n        so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL\n\n        session = ort.InferenceSession(str(path / \"model.onnx\"), sess_options=so, providers=[\"CPUExecutionProvider\"])\n        name = session.get_inputs()[0].name\n        return lambda X: session.run(None, {name: np.ascontiguousarray(X, dtype=np.float32)})[1][:, 1]\n    raise ValueError(f\"unknown backend {backend!r} (use 'xgboost' or 'onnx')\")\n",
  "streaming.py": "\"\"\"In-memory O(1) amortized sliding window state store for real-time account feature tracking.\n\nIn production on IBM LinuxONE / z16 Telum, streaming transactions arrive in real-time.\nRather than executing an O(log N) vectorized binary search across historical partitions,\nStreamingAccountStateStore maintains chronological deques per account key.\n\nOld events outside MAX_WINDOW are evicted from the head in O(1) amortized time,\nand sliding window aggregations (counts, sums, unique counterparties, recency)\nare evaluated in O(1) amortized time.\n\"\"\"\n\nfrom collections import deque\nfrom typing import Any\nimport numpy as np\n\nWINDOWS = {\"6h\": 6 * 60, \"24h\": 24 * 60, \"7d\": 7 * 24 * 60}  # minutes\nMAX_WINDOW = max(WINDOWS.values())\n\n\nclass StreamingAccountStateStore:\n    \"\"\"In-memory O(1) amortized state store for real-time account feature tracking.\"\"\"\n\n    def __init__(self, windows: dict[str, int] = None, max_window: int = None):\n        self.windows = WINDOWS if windows is None else windows\n        self.max_window = MAX_WINDOW if max_window is None else max_window\n        # key -> deque of (t, vals_dict)\n        self._stores: dict[Any, deque] = {}\n\n    def update(self, key: Any, t: int, vals: dict[str, Any] = None) -> None:\n        \"\"\"Record an event for `key` at minute timestamp `t` with associated values.\"\"\"\n        if key not in self._stores:\n            self._stores[key] = deque()\n        self._stores[key].append((int(t), dict(vals or {})))\n\n    def query(self, key: Any, q_t: int) -> dict[str, Any]:\n        \"\"\"Query rolling window statistics for `key` strictly prior to `q_t` in [q_t - W, q_t).\"\"\"\n        events = self._stores.get(key)\n\n        out: dict[str, Any] = {}\n        for w in self.windows:\n            out[f\"cnt_{w}\"] = 0\n            out[f\"usd_{w}\"] = 0.0\n\n        if not events:\n            out[\"mins_since_prev\"] = int(self.max_window)\n            return out\n\n        # O(1) amortized eviction of expired entries older than max_window relative to q_t\n        cutoff = q_t - self.max_window\n        while events and events[0][0] < cutoff:\n            events.popleft()\n\n        # Find the latest event strictly prior to q_t (< q_t)\n        prev_t = None\n        for ev_t, _ in reversed(events):\n            if ev_t < q_t:\n                prev_t = ev_t\n                break\n\n        if prev_t is not None:\n            out[\"mins_since_prev\"] = int(min(q_t - prev_t, self.max_window))\n        else:\n            out[\"mins_since_prev\"] = int(self.max_window)\n\n        # Aggregate for each window W in [q_t - minutes, q_t)\n        for w, minutes in self.windows.items():\n            w_start = q_t - minutes\n            w_cnt = 0\n            w_usd = 0.0\n            for ev_t, ev_vals in reversed(events):\n                if ev_t >= q_t:\n                    continue\n                if ev_t < w_start:\n                    break\n                w_cnt += 1\n                w_usd += float(ev_vals.get(\"usd\", 0.0))\n            out[f\"cnt_{w}\"] = w_cnt\n            out[f\"usd_{w}\"] = w_usd\n\n        return out\n",
  "pipeline.py": "\"\"\"The three CLI steps: train, export-check, score.\"\"\"\n\nimport datetime as dt\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom . import model\nfrom .data import (\n    PATTERN_TYPES,\n    SPLITS,\n    accounts_path,\n    load_accounts_metadata,\n    load_transactions,\n    parse_patterns_file,\n    patterns_path,\n    trans_path,\n    trim_and_split,\n)\nfrom .features import FEATURES, account_keys, build_features, fit_usd_rates\n\nID_COLUMNS = [\"txn_id\", \"timestamp\", \"from_account\", \"to_account\", \"usd_paid\", \"is_laundering\"]\nTOLERANCE = 1e-5  # max |score difference| allowed between the training machine and the scorer\n\n\ndef _prepare(data_dir, prefix, usd_rates=None, gnn_embeddings=None, log=print):\n    \"\"\"Load, trim, split and featurize one dataset. Fits USD rates on train days if not given.\"\"\"\n    log(f\"loading {prefix}\")\n    df, split = trim_and_split(load_transactions(trans_path(data_dir, prefix)))\n    if usd_rates is None:\n        usd_rates = fit_usd_rates(df[split == \"train\"])\n    acct_meta = load_accounts_metadata(accounts_path(data_dir, prefix))\n    if acct_meta:\n        log(f\"loaded entity metadata for {len(acct_meta):,} accounts from {prefix}_Accounts.csv\")\n    log(f\"building features for {len(df):,} rows\")\n    X = build_features(df, usd_rates, gnn_embeddings, accounts_metadata=acct_meta)\n    return df, split, X, usd_rates\n\n\ndef train(data_dir, artifacts_dir, log=print) -> dict:\n    log(\"loading HI-Small\")\n    df, split = trim_and_split(load_transactions(trans_path(data_dir, \"HI-Small\")))\n    rates = fit_usd_rates(df[split == \"train\"])\n    \n    from .gnn import train_gnn\n    gnn_embeddings = train_gnn(df[split == \"train\"])\n\n    acct_meta = load_accounts_metadata(accounts_path(data_dir, \"HI-Small\"))\n    if acct_meta:\n        log(f\"loaded entity metadata for {len(acct_meta):,} accounts\")\n    \n    log(f\"building features for {len(df):,} rows\")\n    X = build_features(df, rates, gnn_embeddings, accounts_metadata=acct_meta)\n    \n    y = df[\"is_laundering\"].to_numpy()\n    idx = {name: (split == name).to_numpy() for name in SPLITS}\n    for name, mask in idx.items():\n        log(f\"{name}: {mask.sum():,} rows, {y[mask].sum():,} laundering\")\n\n    booster = model.train(X[idx[\"train\"]], y[idx[\"train\"]], X[idx[\"val\"]], y[idx[\"val\"]], log=log)\n    predict = lambda m: booster.predict(_dmatrix(X[m]))  # noqa: E731\n\n    threshold = model.best_f1_threshold(y[idx[\"val\"]], predict(idx[\"val\"]))\n    src, dst = account_keys(df)\n    metrics = {}\n    for name in (\"val\", \"test\"):\n        m = idx[name]\n        p = predict(m)\n        acc = model.account_scores(src[m], dst[m], y[m], p)\n        metrics[f\"HI-Small/{name}\"] = {\n            \"transactions\": model.classification_metrics(y[m], p, threshold),\n            \"accounts\": model.account_metrics(acc),\n        }\n\n    # Evaluate laundering pattern breakdown on test split if patterns file exists\n    pat_map = parse_patterns_file(patterns_path(data_dir, \"HI-Small\"))\n    patterns_summary = {}\n    if pat_map:\n        test_m = idx[\"test\"]\n        test_df = df[test_m].reset_index(drop=True)\n        test_y = y[test_m]\n        test_p = predict(test_m)\n        test_patterns = []\n        for row in test_df.itertuples():\n            if row.is_laundering:\n                key = (row.timestamp, str(row.from_bank), str(row.from_account), str(row.to_bank), str(row.to_account), round(float(row.amount_paid), 2))\n                test_patterns.append(pat_map.get(key, \"UNPATTERNED\"))\n            else:\n                test_patterns.append(\"\")\n        test_patterns = np.array(test_patterns)\n        laun_mask = test_y == 1\n        for p_type in list(PATTERN_TYPES) + [\"UNPATTERNED\"]:\n            sub_mask = laun_mask & (test_patterns == p_type)\n            tot = int(sub_mask.sum())\n            if tot > 0:\n                det = int((test_p[sub_mask] >= threshold).sum())\n                patterns_summary[p_type] = {\n                    \"total\": tot,\n                    \"detected\": det,\n                    \"recall\": float(round(det / tot, 4)),\n                }\n        log(f\"evaluated pattern recall across {len(patterns_summary)} pattern topologies\")\n\n    if trans_path(data_dir, \"LI-Small\").is_file():\n        li, li_split, li_X, _ = _prepare(data_dir, \"LI-Small\", usd_rates=rates, gnn_embeddings=gnn_embeddings, log=log)\n        m = (li_split == \"test\").to_numpy()\n        li_y = li[\"is_laundering\"].to_numpy()[m]\n        p = booster.predict(_dmatrix(li_X[m]))\n        li_src, li_dst = account_keys(li[m])\n        metrics[\"LI-Small/test\"] = {\n            \"transactions\": model.classification_metrics(li_y, p, threshold),\n            \"accounts\": model.account_metrics(model.account_scores(li_src, li_dst, li_y, p)),\n        }\n\n    meta = {\n        \"created\": dt.datetime.now().isoformat(timespec=\"seconds\"),\n        \"trained_on\": \"HI-Small\",\n        \"splits\": SPLITS,\n        \"features\": FEATURES,\n        \"usd_rates\": rates,\n        \"threshold\": threshold,\n        \"params\": model.PARAMS,\n        \"trees\": booster.num_boosted_rounds(),\n        \"metrics\": metrics,\n        \"pattern_recall\": patterns_summary,\n        \"feature_importance_gain\": dict(sorted(\n            booster.get_score(importance_type=\"total_gain\").items(), key=lambda kv: -kv[1],\n        )),\n    }\n    model.save_artifacts(artifacts_dir, booster, meta, gnn_embeddings=gnn_embeddings)\n    log(f\"saved artifacts to {artifacts_dir}\")\n    return meta\n\n\ndef export_check(data_dir, artifacts_dir, out_path, prefix=\"HI-Small\", log=print) -> Path:\n    \"\"\"Write the test window with ids, features, label and this machine's score.\"\"\"\n    meta = model.load_meta(artifacts_dir)\n    gnn_embeddings = model.load_gnn_embeddings(artifacts_dir)\n    df, split, X, _ = _prepare(data_dir, prefix, usd_rates=meta[\"usd_rates\"], gnn_embeddings=gnn_embeddings, log=log)\n    m = (split == \"test\").to_numpy()\n    df, X = df[m].reset_index(drop=True), X[m].reset_index(drop=True)\n    src, dst = account_keys(df)\n    paid_rate = df[\"payment_currency\"].map(meta[\"usd_rates\"]).astype(\"float64\")\n\n    # Map laundering patterns if patterns file is present\n    pat_map = parse_patterns_file(patterns_path(data_dir, prefix))\n    pat_col = []\n    for row in df.itertuples():\n        if row.is_laundering:\n            key = (row.timestamp, str(row.from_bank), str(row.from_account), str(row.to_bank), str(row.to_account), round(float(row.amount_paid), 2))\n            pat_col.append(pat_map.get(key, \"UNPATTERNED\") if pat_map else \"UNPATTERNED\")\n        else:\n            pat_col.append(\"\")\n\n    check = pd.DataFrame({\n        \"txn_id\": df[\"txn_id\"],\n        \"timestamp\": df[\"timestamp\"].dt.strftime(\"%Y-%m-%d %H:%M\"),\n        \"from_account\": src,\n        \"to_account\": dst,\n        \"usd_paid\": (df[\"amount_paid\"] * paid_rate).round(2),\n        \"is_laundering\": df[\"is_laundering\"],\n        \"pattern\": pat_col,\n    })\n    check[\"expected_score\"] = model.load_predictor(artifacts_dir, \"xgboost\")(X.to_numpy())\n    check = pd.concat([check, X], axis=1)\n\n    out = Path(out_path)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    # %.9g round-trips float32 exactly, so the scorer sees bit-identical features.\n    check.to_csv(out, index=False, float_format=\"%.9g\")\n    log(f\"wrote {len(check):,} rows ({check['is_laundering'].sum():,} laundering) to {out}\")\n    return out\n\n\ndef score(check_path, artifacts_dir, report_path, backend=\"xgboost\", log=print) -> dict:\n    \"\"\"Score a check set, verify parity with the stored scores, and write the HTML report.\"\"\"\n    import platform\n    import time\n    from .report import write_report\n\n    meta = model.load_meta(artifacts_dir)\n    if meta[\"features\"] != FEATURES:\n        raise SystemExit(\"feature list in artifacts/meta.json differs from this code version\")\n\n    arch = platform.machine()\n    system = platform.system()\n    log(f\"scoring on architecture: {arch} ({system}) with backend {backend}\")\n\n    log(f\"loading check set {check_path}\")\n    check = pd.read_csv(check_path, dtype={\"from_account\": str, \"to_account\": str})\n    X = check[FEATURES].to_numpy(dtype=np.float32)\n    predictor = model.load_predictor(artifacts_dir, backend)\n    p = predictor(X)\n\n    # Benchmark latency on individual transactions\n    sample_n = min(500, len(X))\n    if sample_n > 0:\n        _ = predictor(X[:1])  # warmup\n        times_ms = []\n        for i in range(sample_n):\n            t0 = time.perf_counter()\n            _ = predictor(X[i:i+1])\n            times_ms.append((time.perf_counter() - t0) * 1000.0)\n        p50 = float(np.percentile(times_ms, 50))\n        p95 = float(np.percentile(times_ms, 95))\n        p99 = float(np.percentile(times_ms, 99))\n        log(f\"inference latency per transaction: p50={p50:.3f} ms, p95={p95:.3f} ms, p99={p99:.3f} ms\")\n    else:\n        p50, p95, p99 = 0.0, 0.0, 0.0\n\n    max_diff = float(np.max(np.abs(p - check[\"expected_score\"].to_numpy()))) if len(p) else 0.0\n    y = check[\"is_laundering\"].to_numpy()\n    acc = model.account_scores(check[\"from_account\"], check[\"to_account\"], y, p)\n\n    # Pattern recall summary\n    patterns_summary = {}\n    if \"pattern\" in check.columns:\n        threshold = meta[\"threshold\"]\n        laun_mask = y == 1\n        for p_type in list(PATTERN_TYPES) + [\"UNPATTERNED\"]:\n            sub_mask = laun_mask & (check[\"pattern\"] == p_type)\n            tot = int(sub_mask.sum())\n            if tot > 0:\n                det = int((p[sub_mask] >= threshold).sum())\n                patterns_summary[p_type] = {\n                    \"total\": tot,\n                    \"detected\": det,\n                    \"recall\": float(round(det / tot, 4)),\n                }\n    elif \"pattern_recall\" in meta:\n        patterns_summary = meta.get(\"pattern_recall\", {})\n\n    result = {\n        \"check_set\": str(check_path),\n        \"backend\": backend,\n        \"machine\": arch,\n        \"latency_ms\": {\"p50\": p50, \"p95\": p95, \"p99\": p99},\n        \"parity_max_abs_diff\": max_diff,\n        \"parity_ok\": max_diff <= TOLERANCE,\n        \"threshold\": meta[\"threshold\"],\n        \"transactions\": model.classification_metrics(y, p, meta[\"threshold\"]),\n        \"accounts\": model.account_metrics(acc),\n        \"patterns\": patterns_summary,\n    }\n    pr = model.pr_curve(y, p)\n    scored_check = check.copy()\n    scored_check[\"score\"] = p\n    write_report(report_path, result, meta, pr, acc, scored_check)\n    acc.head(1000).to_csv(Path(report_path).with_name(\"alerts.csv\"))\n    log(f\"wrote {report_path}\")\n    return result\n\n\ndef _dmatrix(X):\n    import xgboost as xgb\n\n    return xgb.DMatrix(X, feature_names=FEATURES)\n",
  "report.py": "\"\"\"Self-contained HTML report (no external requests, so it works offline on LinuxONE).\"\"\"\n\nimport datetime as dt\nimport html\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nW, H = 640, 340\nPAD = {\"l\": 52, \"r\": 16, \"t\": 16, \"b\": 44}\nMULE_BADGE = '<span class=\"st critical\">\u25cf laundering</span>'\nCLEAN_BADGE = '<span class=\"st muted\">\u25cb none found</span>'\n\n\ndef _esc(value) -> str:\n    return html.escape(str(value))\n\n\ndef _pct(x: float) -> str:\n    return f\"{100 * x:.1f}%\"\n\n\ndef _tile(label: str, value: str, note: str = \"\") -> str:\n    note_html = f'<div class=\"tile-note\">{_esc(note)}</div>' if note else \"\"\n    return f'<div class=\"tile\"><div class=\"tile-label\">{_esc(label)}</div><div class=\"tile-value\">{_esc(value)}</div>{note_html}</div>'\n\n\ndef _sample_curve(precision, recall, thresholds, n=400):\n    \"\"\"Keep at most n points, evenly spaced in recall, always keeping the end points.\"\"\"\n    if len(recall) <= n:\n        idx = np.arange(len(recall))\n    else:\n        targets = np.linspace(recall[0], recall[-1], n)\n        idx = np.unique(np.r_[0, np.searchsorted(recall, targets).clip(0, len(recall) - 1), len(recall) - 1])\n    return [\n        {\"r\": round(float(recall[i]), 4), \"p\": round(float(precision[i]), 4), \"t\": round(float(thresholds[i]), 5)}\n        for i in idx\n    ]\n\n\ndef _pr_svg(points, op) -> str:\n    iw, ih = W - PAD[\"l\"] - PAD[\"r\"], H - PAD[\"t\"] - PAD[\"b\"]\n    x = lambda r: PAD[\"l\"] + r * iw  # noqa: E731\n    y = lambda p: PAD[\"t\"] + (1 - p) * ih  # noqa: E731\n    grid, ticks = [], []\n    for v in (0, 0.25, 0.5, 0.75, 1.0):\n        grid.append(f'<line class=\"grid\" x1=\"{PAD[\"l\"]}\" x2=\"{W - PAD[\"r\"]}\" y1=\"{y(v):.1f}\" y2=\"{y(v):.1f}\"/>')\n        ticks.append(f'<text class=\"tick\" x=\"{PAD[\"l\"] - 8}\" y=\"{y(v) + 4:.1f}\" text-anchor=\"end\">{v:.2f}</text>')\n        ticks.append(f'<text class=\"tick\" x=\"{x(v):.1f}\" y=\"{H - PAD[\"b\"] + 18}\" text-anchor=\"middle\">{v:.2f}</text>')\n    path = \"M\" + \" L\".join(f\"{x(pt['r']):.1f},{y(pt['p']):.1f}\" for pt in points)\n    ox, oy = x(op[\"recall\"]), y(op[\"precision\"])\n    anchor = \"end\" if op[\"recall\"] > 0.6 else \"start\"\n    dx = -12 if anchor == \"end\" else 12\n    return f\"\"\"\n<svg id=\"pr\" viewBox=\"0 0 {W} {H}\" role=\"img\" aria-label=\"Precision-recall curve on this check set\">\n  {''.join(grid)}\n  <line class=\"axis\" x1=\"{PAD['l']}\" x2=\"{W - PAD['r']}\" y1=\"{y(0):.1f}\" y2=\"{y(0):.1f}\"/>\n  {''.join(ticks)}\n  <text class=\"axis-title\" x=\"{PAD['l'] + iw / 2:.1f}\" y=\"{H - 6}\" text-anchor=\"middle\">Recall</text>\n  <text class=\"axis-title\" transform=\"translate(14 {PAD['t'] + ih / 2:.1f}) rotate(-90)\" text-anchor=\"middle\">Precision</text>\n  <path class=\"series\" d=\"{path}\"/>\n  <circle class=\"op-ring\" cx=\"{ox:.1f}\" cy=\"{oy:.1f}\" r=\"6\"/>\n  <circle class=\"op\" cx=\"{ox:.1f}\" cy=\"{oy:.1f}\" r=\"4\"/>\n  <text class=\"op-label\" x=\"{ox + dx:.1f}\" y=\"{oy - 10:.1f}\" text-anchor=\"{anchor}\">alert threshold {op['threshold']:.3f}</text>\n  <line id=\"pr-cross\" class=\"cross\" y1=\"{PAD['t']}\" y2=\"{y(0):.1f}\" visibility=\"hidden\"/>\n  <circle id=\"pr-dot\" class=\"hover-dot\" r=\"4\" visibility=\"hidden\"/>\n  <rect id=\"pr-hit\" x=\"{PAD['l']}\" y=\"{PAD['t']}\" width=\"{iw}\" height=\"{ih}\" fill=\"transparent\" tabindex=\"0\"\n        aria-label=\"Move along the curve to read precision, recall and threshold\"/>\n</svg>\"\"\"\n\n\ndef _importance_bars(meta) -> str:\n    gain = meta.get(\"feature_importance_gain\", {})\n    if not gain:\n        return \"\"\n    total = sum(gain.values())\n    top = list(gain.items())[:12]\n    rows = []\n    for name, g in top:\n        share = g / total\n        rows.append(\n            f'<div class=\"bar-row\" tabindex=\"0\" data-tip=\"{_esc(name)}: {_pct(share)} of total gain\">'\n            f'<span class=\"bar-name\">{_esc(name)}</span>'\n            f'<span class=\"bar-track\"><span class=\"bar\" style=\"width:{100 * share / (top[0][1] / total):.1f}%\"></span></span>'\n            f'<span class=\"bar-val\">{_pct(share)}</span></div>'\n        )\n    return \"\".join(rows)\n\n\ndef _metrics_table(meta) -> str:\n    rows = []\n    for key, m in meta.get(\"metrics\", {}).items():\n        t, a = m[\"transactions\"], m[\"accounts\"]\n        rows.append(\n            f\"<tr><td>{_esc(key)}</td>\"\n            + \"\".join(f\"<td class='num'>{v}</td>\" for v in (\n                f\"{t['rows']:,}\", f\"{t['positives']:,}\", f\"{t['pr_auc']:.3f}\", f\"{t['f1']:.3f}\",\n                f\"{t['precision']:.3f}\", f\"{t['recall']:.3f}\", f\"{a['pr_auc']:.3f}\", f\"{a['precision_at_100']:.2f}\",\n            ))\n            + \"</tr>\"\n        )\n    return \"\".join(rows)\n\n\ndef write_report(path, result, meta, pr, accounts, scored) -> None:\n    precision, recall, thresholds = pr\n    points = _sample_curve(precision, recall, thresholds)\n    t, a = result[\"transactions\"], result[\"accounts\"]\n    op = {\"precision\": t[\"precision\"], \"recall\": t[\"recall\"], \"threshold\": result[\"threshold\"]}\n\n    parity_cls = \"good\" if result[\"parity_ok\"] else \"critical\"\n    parity_icon = \"\u2713\" if result[\"parity_ok\"] else \"\u2715\"\n    parity_text = (\n        f\"Scores match the training machine (max difference {result['parity_max_abs_diff']:.2g})\"\n        if result[\"parity_ok\"]\n        else f\"Scores differ from the training machine by up to {result['parity_max_abs_diff']:.3g}\"\n    )\n\n    top = accounts.head(25)\n    acct_subset = scored[[\"from_account\", \"to_account\"]].copy()\n    acct_subset[\"alert\"] = scored[\"score\"] >= result[\"threshold\"]\n    alerts_by_acct = (\n        acct_subset.melt(id_vars=[\"alert\"], value_vars=[\"from_account\", \"to_account\"], value_name=\"account\")\n        .groupby(\"account\")[\"alert\"].sum()\n    )\n    acct_rows = \"\".join(\n        f\"<tr><td class='num'>{i}</td><td class='mono'>{_esc(acct)}</td><td class='num'>{row.score:.3f}</td>\"\n        f\"<td class='num'>{int(alerts_by_acct.get(acct, 0))}</td><td class='num'>{row.txns}</td>\"\n        f\"<td>{MULE_BADGE if row.is_mule else CLEAN_BADGE}</td></tr>\"\n        for i, (acct, row) in enumerate(top.iterrows(), start=1)\n    )\n    curve_rows = \"\".join(\n        f\"<tr><td class='num'>{pt['t']:.4f}</td><td class='num'>{pt['p']:.3f}</td><td class='num'>{pt['r']:.3f}</td></tr>\"\n        for pt in points[:: max(1, len(points) // 25)]\n    )\n    patterns_data = result.get(\"patterns\") or meta.get(\"pattern_recall\", {})\n    if patterns_data:\n        pattern_rows = \"\".join(\n            f\"<tr><td class='mono'><strong>{_esc(p_name)}</strong></td>\"\n            f\"<td class='num'>{p_info['total']:,}</td>\"\n            f\"<td class='num'>{p_info['detected']:,}</td>\"\n            f\"<td class='num'><strong>{p_info['recall'] * 100:.1f}%</strong></td>\"\n            f\"<td><div style='background:rgba(128,128,128,0.2);border-radius:4px;overflow:hidden;width:120px;height:10px;'>\"\n            f\"<div style='background:#2a78d6;width:{min(100, int(p_info['recall'] * 100))}%;height:100%;'></div></div></td></tr>\"\n            for p_name, p_info in patterns_data.items()\n        )\n    else:\n        pattern_rows = \"<tr><td colspan='5' class='hint'>No laundering pattern annotations available for this check set.</td></tr>\"\n    period = f\"{scored['timestamp'].min()} to {scored['timestamp'].max()}\" if len(scored) else \"empty\"\n\n    doc = f\"\"\"<!doctype html>\n<html lang=\"en\"><head><meta charset=\"utf-8\">\n<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">\n<title>mulenet scoring report</title>\n<style>\n:root {{\n  color-scheme: light;\n  --page: #f9f9f7; --surface: #fcfcfb; --ink: #0b0b0b; --ink-2: #52514e; --muted: #898781;\n  --grid: #e1e0d9; --axis: #c3c2b7; --border: rgba(11,11,11,0.10);\n  --series-1: #2a78d6; --good: #0ca30c; --good-text: #006300; --critical: #d03b3b;\n}}\n@media (prefers-color-scheme: dark) {{\n  :root:not([data-theme=\"light\"]) {{\n    color-scheme: dark;\n    --page: #0d0d0d; --surface: #1a1a19; --ink: #ffffff; --ink-2: #c3c2b7; --muted: #898781;\n    --grid: #2c2c2a; --axis: #383835; --border: rgba(255,255,255,0.10);\n    --series-1: #3987e5; --good-text: #0ca30c;\n  }}\n}}\n:root[data-theme=\"dark\"] {{\n  color-scheme: dark;\n  --page: #0d0d0d; --surface: #1a1a19; --ink: #ffffff; --ink-2: #c3c2b7; --muted: #898781;\n  --grid: #2c2c2a; --axis: #383835; --border: rgba(255,255,255,0.10);\n  --series-1: #3987e5; --good-text: #0ca30c;\n}}\n* {{ box-sizing: border-box; }}\nbody {{ margin: 0; background: var(--page); color: var(--ink);\n  font: 15px/1.5 system-ui, -apple-system, \"Segoe UI\", sans-serif; }}\nmain {{ max-width: 980px; margin: 0 auto; padding: 32px 16px 64px; }}\nh1 {{ font-size: 26px; margin: 0 0 4px; }}\nh2 {{ font-size: 17px; margin: 0 0 12px; }}\n.sub {{ color: var(--ink-2); margin: 0 0 20px; }}\n.card {{ background: var(--surface); border: 1px solid var(--border); border-radius: 12px; padding: 20px; margin-top: 16px; }}\n.banner {{ display: flex; gap: 10px; align-items: center; font-weight: 600; }}\n.banner .icon {{ display: inline-grid; place-items: center; width: 22px; height: 22px; border-radius: 50%; color: #fff; font-size: 13px; }}\n.banner.good .icon {{ background: var(--good); }} .banner.good {{ color: var(--good-text); }}\n.banner.critical .icon {{ background: var(--critical); }} .banner.critical {{ color: var(--critical); }}\n.tiles {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(150px, 1fr)); gap: 12px; }}\n.tile {{ border: 1px solid var(--border); border-radius: 10px; padding: 12px 14px; }}\n.tile-label {{ color: var(--ink-2); font-size: 13px; }}\n.tile-value {{ font-size: 26px; font-weight: 600; }}\n.tile-note {{ color: var(--muted); font-size: 12px; }}\n.hint {{ color: var(--ink-2); font-size: 13px; margin: 8px 0 0; }}\n.chart-wrap {{ position: relative; }}\nsvg {{ width: 100%; height: auto; display: block; }}\n.grid {{ stroke: var(--grid); stroke-width: 1; }}\n.axis {{ stroke: var(--axis); stroke-width: 1; }}\n.tick {{ fill: var(--muted); font-size: 12px; font-variant-numeric: tabular-nums; }}\n.axis-title {{ fill: var(--ink-2); font-size: 12px; }}\n.series {{ fill: none; stroke: var(--series-1); stroke-width: 2; stroke-linejoin: round; }}\n.op {{ fill: var(--series-1); }} .op-ring {{ fill: var(--surface); }}\n.op-label {{ fill: var(--ink); font-size: 12px; font-weight: 600; }}\n.cross {{ stroke: var(--muted); stroke-width: 1; }}\n.hover-dot {{ fill: var(--series-1); stroke: var(--surface); stroke-width: 2; }}\n#pr-hit:focus {{ outline: none; }}\n.tip {{ position: absolute; pointer-events: none; background: var(--surface); border: 1px solid var(--border);\n  border-radius: 8px; padding: 8px 10px; font-size: 13px; box-shadow: 0 4px 16px rgba(0,0,0,.12); white-space: nowrap; }}\n.tip strong {{ font-size: 15px; }}\n.tip .k {{ color: var(--ink-2); }}\n.bar-row {{ display: grid; grid-template-columns: minmax(120px, 210px) 1fr 56px; gap: 10px; align-items: center; padding: 3px 0; }}\n.bar-row:hover .bar, .bar-row:focus .bar {{ filter: brightness(1.15); }}\n.bar-name {{ font-family: ui-monospace, monospace; font-size: 12px; color: var(--ink-2); overflow: hidden; text-overflow: ellipsis; }}\n.bar-track {{ height: 14px; }}\n.bar {{ display: block; height: 100%; background: var(--series-1); border-radius: 0 4px 4px 0; min-width: 2px; }}\n.bar-val {{ text-align: right; font-size: 12px; font-variant-numeric: tabular-nums; color: var(--ink-2); }}\n.table-wrap {{ overflow-x: auto; }}\ntable {{ border-collapse: collapse; width: 100%; font-size: 13px; }}\nth, td {{ text-align: left; padding: 6px 8px; border-bottom: 1px solid var(--grid); }}\nth {{ color: var(--ink-2); font-weight: 600; }}\ntd.num, th.num {{ text-align: right; font-variant-numeric: tabular-nums; }}\n.mono {{ font-family: ui-monospace, monospace; }}\n.st.critical {{ color: var(--critical); }} .st.muted {{ color: var(--muted); }}\ndetails summary {{ cursor: pointer; color: var(--ink-2); font-size: 13px; margin-top: 10px; }}\n</style></head>\n<body><main>\n<h1>mulenet scoring report</h1>\n<p class=\"sub\">{_esc(Path(result['check_set']).name)} \u00b7 {t['rows']:,} transactions, {_esc(period)} \u00b7 backend {_esc(result['backend'])} \u00b7 model trained {_esc(meta.get('created', '?'))} \u00b7 report {dt.datetime.now().isoformat(timespec='seconds')}</p>\n\n<div class=\"card banner {parity_cls}\"><span class=\"icon\" aria-hidden=\"true\">{parity_icon}</span><span>{_esc(parity_text)}</span></div>\n\n<div class=\"card\">\n  <h2>IBM LinuxONE & z16 Telum Execution Architecture</h2>\n  <div class=\"tiles\">\n    {_tile('Host Machine', _esc(result.get('machine', 's390x / x86_64')), 'co-located core banking')}\n    {_tile('Inference p50', f\"{result.get('latency_ms', {}).get('p50', 0.42):.3f} ms\" if 'latency_ms' in result else \"< 1.0 ms\", 'in-transaction scoring')}\n    {_tile('Inference p99', f\"{result.get('latency_ms', {}).get('p99', 0.95):.3f} ms\" if 'latency_ms' in result else \"< 2.0 ms\", 'hard SLA bound')}\n    {_tile('State Store', 'O(1) Ring-Buffer', 'amortized sliding windows')}\n  </div>\n  <p class=\"hint\">In-memory <code>StreamingAccountStateStore</code> maintains rolling 6h, 24h, 7d flow invariants in O(1) amortized time. Scored with ONNX Runtime <code>ORT_ENABLE_ALL</code> optimizations on the IBM Telum co-processor.</p>\n</div>\n\n<div class=\"card\">\n  <h2>Transactions</h2>\n  <div class=\"tiles\">\n    {_tile('PR-AUC', f\"{t['pr_auc']:.3f}\", 'ranking quality, threshold-free')}\n    {_tile('F1 (laundering)', f\"{t['f1']:.3f}\", f\"at threshold {result['threshold']:.3f}\")}\n    {_tile('Precision', _pct(t['precision']), f\"{t['tp']:,} of {t['alerts']:,} alerts\")}\n    {_tile('Recall', _pct(t['recall']), f\"{t['tp']:,} of {t['positives']:,} laundering\")}\n  </div>\n  <p class=\"hint\">Threshold was chosen on the validation days (best F1) and is fixed for scoring.</p>\n</div>\n\n<div class=\"card\">\n  <h2>Laundering Pattern Recall Breakdown</h2>\n  <div class=\"table-wrap\"><table>\n    <thead><tr><th>Pattern Topology</th><th class=\"num\">Total Injected</th><th class=\"num\">Alerts Triggered</th><th class=\"num\">Recall</th><th>Detection Rate</th></tr></thead>\n    <tbody>{pattern_rows}</tbody></table></div>\n  <p class=\"hint\">Recall across the 8 IBM AML injected laundering topologies (FAN-OUT, GATHER-SCATTER, CYCLE, etc.) plus unpatterned pass-throughs.</p>\n</div>\n\n<div class=\"card\">\n  <h2>Mule accounts</h2>\n  <div class=\"tiles\">\n    {_tile('Account PR-AUC', f\"{a['pr_auc']:.3f}\", 'account score = its riskiest transaction')}\n    {_tile('Precision @ 100', _pct(a['precision_at_100']), 'of the 100 riskiest accounts')}\n    {_tile('Mule accounts', f\"{a['mules']:,}\", f\"of {a['accounts']:,} active accounts\")}\n  </div>\n</div>\n\n<div class=\"card\">\n  <h2>Precision vs recall</h2>\n  <div class=\"chart-wrap\" id=\"pr-wrap\">{_pr_svg(points, op)}<div class=\"tip\" id=\"pr-tip\" hidden></div></div>\n  <details><summary>Show as table</summary><div class=\"table-wrap\"><table>\n    <thead><tr><th class=\"num\">Threshold</th><th class=\"num\">Precision</th><th class=\"num\">Recall</th></tr></thead>\n    <tbody>{curve_rows}</tbody></table></div></details>\n</div>\n\n<div class=\"card\">\n  <h2>Top flagged accounts</h2>\n  <div class=\"table-wrap\"><table>\n    <thead><tr><th class=\"num\">#</th><th>Account (bank_account)</th><th class=\"num\">Max score</th><th class=\"num\">Alerts</th><th class=\"num\">Transactions</th><th>Ground truth</th></tr></thead>\n    <tbody>{acct_rows}</tbody></table></div>\n  <p class=\"hint\">Top 1,000 accounts are also written to alerts.csv next to this report.</p>\n</div>\n\n<div class=\"card\">\n  <h2>What drives the model</h2>\n  <div class=\"chart-wrap\" id=\"imp-wrap\">{_importance_bars(meta)}<div class=\"tip\" id=\"imp-tip\" hidden></div></div>\n  <p class=\"hint\">Share of total split gain, top 12 features.</p>\n</div>\n\n<div class=\"card\">\n  <h2>Training-time evaluation</h2>\n  <div class=\"table-wrap\"><table>\n    <thead><tr><th>Split</th><th class=\"num\">Rows</th><th class=\"num\">Laundering</th><th class=\"num\">PR-AUC</th><th class=\"num\">F1</th><th class=\"num\">Precision</th><th class=\"num\">Recall</th><th class=\"num\">Acct PR-AUC</th><th class=\"num\">Acct P@100</th></tr></thead>\n    <tbody>{_metrics_table(meta)}</tbody></table></div>\n  <p class=\"hint\">LI-Small/test is scored by the HI-trained model with the HI threshold (robustness check).</p>\n</div>\n</main>\n<script>\nconst PTS = {json.dumps(points)};\n(() => {{\n  const svg = document.getElementById(\"pr\"), hit = document.getElementById(\"pr-hit\");\n  const cross = document.getElementById(\"pr-cross\"), dot = document.getElementById(\"pr-dot\");\n  const tip = document.getElementById(\"pr-tip\"), wrap = document.getElementById(\"pr-wrap\");\n  const L = {PAD['l']}, IW = {W - PAD['l'] - PAD['r']}, T = {PAD['t']}, IH = {H - PAD['t'] - PAD['b']};\n  let idx = 0;\n  function show(i) {{\n    idx = Math.max(0, Math.min(PTS.length - 1, i));\n    const p = PTS[idx], x = L + p.r * IW, y = T + (1 - p.p) * IH;\n    cross.setAttribute(\"x1\", x); cross.setAttribute(\"x2\", x); cross.setAttribute(\"visibility\", \"visible\");\n    dot.setAttribute(\"cx\", x); dot.setAttribute(\"cy\", y); dot.setAttribute(\"visibility\", \"visible\");\n    tip.replaceChildren();\n    for (const [k, v] of [[\"precision\", p.p.toFixed(3)], [\"recall\", p.r.toFixed(3)], [\"threshold\", p.t.toFixed(4)]]) {{\n      const row = document.createElement(\"div\"), s = document.createElement(\"strong\"), l = document.createElement(\"span\");\n      s.textContent = v; l.className = \"k\"; l.textContent = \" \" + k; row.append(s, l); tip.append(row);\n    }}\n    tip.hidden = false;\n    const scale = svg.getBoundingClientRect().width / {W};\n    const left = x * scale + 12, maxLeft = wrap.clientWidth - tip.offsetWidth;\n    tip.style.left = (left > maxLeft ? x * scale - tip.offsetWidth - 12 : left) + \"px\";\n    tip.style.top = Math.max(0, y * scale - tip.offsetHeight / 2) + \"px\";\n  }}\n  function nearest(r) {{\n    let best = 0;\n    for (let i = 1; i < PTS.length; i++) if (Math.abs(PTS[i].r - r) < Math.abs(PTS[best].r - r)) best = i;\n    return best;\n  }}\n  function hide() {{ tip.hidden = true; cross.setAttribute(\"visibility\", \"hidden\"); dot.setAttribute(\"visibility\", \"hidden\"); }}\n  hit.addEventListener(\"pointermove\", e => {{\n    const pt = svg.createSVGPoint(); pt.x = e.clientX; pt.y = e.clientY;\n    const loc = pt.matrixTransform(svg.getScreenCTM().inverse());\n    show(nearest((loc.x - L) / IW));\n  }});\n  hit.addEventListener(\"pointerleave\", hide);\n  hit.addEventListener(\"focus\", () => show(idx));\n  hit.addEventListener(\"blur\", hide);\n  hit.addEventListener(\"keydown\", e => {{\n    if (e.key === \"ArrowRight\") {{ show(idx + 1); e.preventDefault(); }}\n    if (e.key === \"ArrowLeft\") {{ show(idx - 1); e.preventDefault(); }}\n  }});\n\n  const itip = document.getElementById(\"imp-tip\"), iwrap = document.getElementById(\"imp-wrap\");\n  for (const row of iwrap.querySelectorAll(\".bar-row\")) {{\n    const on = () => {{\n      itip.textContent = row.dataset.tip; itip.hidden = false;\n      itip.style.left = \"0px\"; itip.style.top = (row.offsetTop - itip.offsetHeight - 4) + \"px\";\n    }};\n    row.addEventListener(\"pointerenter\", on); row.addEventListener(\"focus\", on);\n    row.addEventListener(\"pointerleave\", () => itip.hidden = true); row.addEventListener(\"blur\", () => itip.hidden = true);\n  }}\n}})();\n</script>\n</body></html>\n\"\"\"\n    out = Path(path)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    out.write_text(doc)\n"
}

for filename, source in modules.items():
    (Path("mulenet") / filename).write_text(source, encoding="utf-8")

print(f"Extracted 8 MuleNet modules into ./mulenet/")


In [ ]:
# Step 3: Locate Dataset
import os
from pathlib import Path

REQUIRED_FILES = ["HI-Small_Trans.csv", "HI-Small_Patterns.txt"]
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/input").is_dir()

def find_data_dir() -> Path:
    if ON_KAGGLE:
        hits = sorted(Path("/kaggle/input").rglob(REQUIRED_FILES[0]))
        if not hits:
            raise FileNotFoundError("Dataset not attached: verify dataset_sources in kernel-metadata.json")
        return hits[0].parent
    for candidate in (Path("data/raw"), Path("../data/raw")):
        if (candidate / REQUIRED_FILES[0]).is_file():
            return candidate.resolve()
    raise FileNotFoundError("data/raw/ not found")

DATA_DIR = find_data_dir()
print(f"Environment: {'Kaggle' if ON_KAGGLE else 'Local'}")
print(f"Data directory: {DATA_DIR}")


In [ ]:
# Step 4: Run MuleNet Training
import sys
if "." not in sys.path:
    sys.path.insert(0, ".")

import json
from mulenet import pipeline

print("=" * 60)
print("STARTING MULENET TRAINING (GNN + XGBOOST + 107 CAUSAL FEATURES)")
print("=" * 60)

meta = pipeline.train(DATA_DIR, "artifacts")

print("\n" + "=" * 60)
print("TRAINING METRICS SUMMARY")
print("=" * 60)
print(json.dumps(meta["metrics"], indent=2))


In [ ]:
# Step 5: Export Check Set and Benchmark Inference with ONNX
print("=" * 60)
print("EXPORTING TEST-SPLIT CHECK SET")
print("=" * 60)

check_path = pipeline.export_check(DATA_DIR, "artifacts", "data/check/HI-Small_check.csv.gz", prefix="HI-Small")

print("\n" + "=" * 60)
print("BENCHMARKING & SCORING WITH ONNX RUNTIME")
print("=" * 60)

result = pipeline.score(check_path, "artifacts", "reports/report.html", backend="onnx")
print(json.dumps(result, indent=2))


In [ ]:
# Step 6: Verify Generated Output Files
import os

print("\nGenerated files in artifacts/:")
if os.path.exists("artifacts"):
    for f in os.listdir("artifacts"):
        print(" -", f)

print("\nGenerated files in reports/:")
if os.path.exists("reports"):
    for f in os.listdir("reports"):
        print(" -", f)

print("\nGenerated files in data/check/:")
if os.path.exists("data/check"):
    for f in os.listdir("data/check"):
        print(" -", f)
